In [1]:
import os, warnings, logging
os.environ["OMP_NUM_THREADS"] = "14"
os.environ["MKL_NUM_THREADS"] = "14"

from pathlib import Path
ROOT = Path("..").resolve()
os.environ["CELLTYPIST_FOLDER"] = str(ROOT / "models")

import scanpy as sc
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    import celltypist
    from celltypist import models

logging.getLogger("celltypist").setLevel(logging.ERROR)

adata = sc.read_h5ad(ROOT / "data/processed/pbmc3k.h5ad")

print("model folder:", Path(models.models_path).relative_to(ROOT))
models.download_models(force_update=False,
                       model=["Immune_All_Low.pkl", "Immune_All_High.pkl"])
print("models ready")

model folder: models/data/models
models ready


In [2]:
# Prepare input for CellTypist

ct_input = adata.copy()
sc.pp.normalize_total(ct_input, target_sum=1e4)
sc.pp.log1p(ct_input)

print("max after log1p:", ct_input.X.max())
print("shape:", ct_input.shape)




max after log1p: 7.4695992
shape: (2638, 13656)


In [3]:
# Run both models

import warnings
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

pred_high = celltypist.annotate(ct_input, model="Immune_All_High.pkl", majority_voting=False)
pred_low  = celltypist.annotate(ct_input, model="Immune_All_Low.pkl",  majority_voting=False)

print("HIGH (coarse):")
print(pred_high.predicted_labels['predicted_labels'].value_counts())
print()
print("LOW (fine):")
print(pred_low.predicted_labels['predicted_labels'].value_counts().head(20))

HIGH (coarse):
predicted_labels
T cells                       1411
Monocytes                      626
B cells                        344
ILC                            188
DC                              26
Megakaryocytes/platelets        12
Macrophages                     11
Myelocytes                       7
pDC                              4
HSC/MPP                          3
Plasma cells                     3
Double-positive thymocytes       1
MNP                              1
Megakaryocyte precursor          1
Name: count, dtype: int64

LOW (fine):
predicted_labels
Tcm/Naive helper T cells       633
Classical monocytes            392
B cells                        340
Non-classical monocytes        229
Tcm/Naive cytotoxic T cells    218
Tem/Effector helper T cells    189
CD16+ NK cells                 149
Tem/Trm cytotoxic T cells      108
Regulatory T cells             103
MAIT cells                      73
NK cells                        57
DC                              20
Te

In [4]:
# Save predictions in the file contract format
import pandas as pd

RES = ROOT / "results"
RES.mkdir(exist_ok=True)

for name, pred in [("celltypist_high", pred_high), ("celltypist_low", pred_low)]:
    df = pd.DataFrame({
        "barcode":    pred.predicted_labels.index,
        "label":      pred.predicted_labels["predicted_labels"].values,
        "confidence": pred.probability_matrix.max(axis=1).values,
    })
    df.to_csv(RES / f"predictions_{name}_pbmc3k.csv", index=False)
    print(name, df.shape, "mean conf:", round(df.confidence.mean(), 3))

celltypist_high (2638, 3) mean conf: 0.972
celltypist_low (2638, 3) mean conf: 0.804
